 Как устроено решение

1. Загружаются `train.csv`, `test.csv` и `events.csv.gz`.
2. Для каждой cookie используются только события внутри её окна:
   `window_start_ts <= event_ts < window_end_ts`.
3. Считается baseline из исходного quickstart.
4. Строятся расширенные поведенческие признаки.
5. Выполняется временная кросс-валидация.
6. Параметры `HistGradientBoostingClassifier` подбираются через `GridSearchCV`.
7. Выбранная модель обучается на всём train.
8. Cоздаётся `submission.csv`.

Основной файл с кодом — `solution.ipynb`.

Test расположен позже train по времени, поэтому случайный `train_test_split` или `StratifiedKFold`
могли бы смешать прошлое и будущее.

Используется временная схема:
- Fold 1: обучение на 6–10 апреля, проверка на 11–13 апреля;
- Fold 2: обучение на 6–13 апреля, проверка на 14–16 апреля.

Период 17–19 апреля используется отдельно как поздняя диагностическая проверка.

Случайная стратификация не применяется, чтобы сохранить временную структуру задачи.

Для каждой cookie строятся агрегаты по её собственной истории событий.

`HistGradientBoostingClassifier` умеет работать с числовыми `NaN` напрямую,
поэтому пропуски не заполняются средними значениями.

Отсутствие поискового запроса, координат или других полей само по себе может быть информативным.

Строковые категории напрямую в итоговую модель не передаются:
используются их числовые агрегаты, доли и количество уникальных значений.

Baseline из исходного quickstart:
- признаки: `n_events`, `item_nunique`;
- модель: `RandomForestClassifier`;
- `random_state = 0`.

Итоговая модель — `HistGradientBoostingClassifier`.

Почему она выбрана:
- данные после агрегации табличные и числовые;
- зависимости могут быть нелинейными;
- деревья находят взаимодействия между признаками;
- стандартизация не нужна;
- поддерживаются `NaN`;
- модель подходит для перебора гиперпараметров.

Параметры выбираются через `GridSearchCV`.

Перебиралась сетка:

```python
PARAM_GRID = {
    'max_leaf_nodes': [3, 7, 15],
    'min_samples_leaf': [10, 20, 40],
    'l2_regularization': [0, 5, 20],
    'learning_rate': [0.03, 0.05],
    'max_iter': [300, 500, 800],
}
```

Всего — 162 комбинации.

Лучшая конфигурация:

```python
HistGradientBoostingClassifier(
    max_iter=800,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=10,
    l2_regularization=0,
    early_stopping=False,
    random_state=42,
)
```

Используется официальная метрика:

**максимальный Precision среди порогов, где Recall ≥ 70%.**

Порог 0.5 не используется для выбора модели.
В `submission.csv` сохраняется непрерывный `score`.

Для итоговой модели зафиксирован `random_state = 42`.

Лучшие параметры Grid Search сохранены в коде.
По умолчанию решение использует уже выбранную конфигурацию,
поэтому повторный запуск не требует повторения всех 162 вариантов.

После запуска модель обучается на всём train и создаёт новый `submission.csv`.